In [1]:
import os
os.environ['HADOOP_HOME'] = 'C:\\hadoop'
os.environ['PATH'] += os.pathsep + os.path.join(os.environ['HADOOP_HOME'], 'bin')

In [1]:

from pyspark.sql import SparkSession
from pyspark.sql.types import StructType, StructField, StringType, FloatType

from delta import *
# Create SparkSession
spark = (
    SparkSession
    .builder
    .master("local[*]")
    .config("spark.jars.packages", "io.delta:delta-spark_2.12:3.2.0")
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension")
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog")
    .getOrCreate()
)

25/04/24 22:00:02 WARN Utils: Your hostname, Central resolves to a loopback address: 127.0.1.1; using 172.28.38.136 instead (on interface eth0)
25/04/24 22:00:02 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address


:: loading settings :: url = jar:file:/home/taylor/APACHE-SPARK-COM-DELTA-LAKE-E-APACHE-ICEBERG/.venv/lib/python3.12/site-packages/pyspark/jars/ivy-2.5.1.jar!/org/apache/ivy/core/settings/ivysettings.xml


Ivy Default Cache set to: /home/taylor/.ivy2/cache
The jars for the packages stored in: /home/taylor/.ivy2/jars
io.delta#delta-spark_2.12 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-258b9aa7-bedf-4f3e-802a-d2647593392e;1.0
	confs: [default]
	found io.delta#delta-spark_2.12;3.2.0 in central
	found io.delta#delta-storage;3.2.0 in central
	found org.antlr#antlr4-runtime;4.9.3 in central
downloading https://repo1.maven.org/maven2/io/delta/delta-spark_2.12/3.2.0/delta-spark_2.12-3.2.0.jar ...
	[SUCCESSFUL ] io.delta#delta-spark_2.12;3.2.0!delta-spark_2.12.jar (1331ms)
downloading https://repo1.maven.org/maven2/io/delta/delta-storage/3.2.0/delta-storage-3.2.0.jar ...
	[SUCCESSFUL ] io.delta#delta-storage;3.2.0!delta-storage.jar (261ms)
downloading https://repo1.maven.org/maven2/org/antlr/antlr4-runtime/4.9.3/antlr4-runtime-4.9.3.jar ...
	[SUCCESSFUL ] org.antlr#antlr4-runtime;4.9.3!antlr4-runtime.jar (285ms)
:: resolution report :: resolve 3864ms :

In [2]:
spark

In [3]:
# Criando tabela de vendas
spark.sql("""
    CREATE TABLE vendas_delta (
        id_venda INT,
        produto STRING,
        quantidade INT,
        valor_unitario DECIMAL(10,2),
        data_venda DATE,
        status STRING
    ) USING delta
""")


DataFrame[]

In [6]:
# Inserindo dados iniciais
spark.sql("""
    INSERT INTO vendas_delta (id_venda, produto, quantidade, valor_unitario, data_venda, status)
    VALUES 
        (1, 'Notebook', 2, 4500.00, '2024-01-15', 'CONCLUIDO'),
        (2, 'Smartphone', 3, 2500.00, '2024-01-16', 'CONCLUIDO'),
        (3, 'Tablet', 1, 1800.00, '2024-01-17', 'PENDENTE'),
        (4, 'Monitor', 2, 1200.00, '2024-01-18', 'CANCELADO')
""")

DataFrame[]

In [ ]:
# Atualizando status e valor de uma venda
spark.sql("""
    UPDATE vendas_delta 
    SET status = 'CONCLUIDO',
        valor_unitario = 1700.00
    WHERE id_venda = 3
""")

In [ ]:

# Deletando vendas canceladas
spark.sql("""
    DELETE FROM vendas_delta 
    WHERE status = 'CANCELADO'
""")

In [9]:
# Consultando vendas com valor total
spark.sql("""
    SELECT 
        id_venda,
        produto,
        quantidade,
        valor_unitario,
        quantidade * valor_unitario as valor_total,
        data_venda,
        status
    FROM vendas_delta
    ORDER BY data_venda
""").show()

+--------+----------+----------+--------------+-----------+----------+---------+
|id_venda|   produto|quantidade|valor_unitario|valor_total|data_venda|   status|
+--------+----------+----------+--------------+-----------+----------+---------+
|       1|  Notebook|         2|       4500.00|    9000.00|2024-01-15|CONCLUIDO|
|       1|  Notebook|         2|       4500.00|    9000.00|2024-01-15|CONCLUIDO|
|       2|Smartphone|         3|       2500.00|    7500.00|2024-01-16|CONCLUIDO|
|       2|Smartphone|         3|       2500.00|    7500.00|2024-01-16|CONCLUIDO|
|       3|    Tablet|         1|       1700.00|    1700.00|2024-01-17|CONCLUIDO|
|       3|    Tablet|         1|       1700.00|    1700.00|2024-01-17|CONCLUIDO|
+--------+----------+----------+--------------+-----------+----------+---------+

